In [19]:
# Jupyter-specific commands to auto-reload external modules and display plots inline
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, sys
import pandas as pd
from typing import cast, Optional

# Ensure project root directory, not the notebooks directory
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Add this new root directory to the path so imports work
sys.path.append(os.getcwd())
project_root = os.getcwd()

print(f"Working Directory set to: {os.getcwd()}")

# Simulation Imports
from src.simulate_fits_data import simulate_fits_data

# Detection Pipeline Imports
from src.process_fits_files import process_fits_directory 

# Tracking Imports
from utils.config_loader import TrackerConfig
from src.tomht import TOMHTTracker
from utils.post_process import (
    plot_longest_tracks,
    animate_tracks,
    print_tomht_stats,
    compute_mota,
    compute_mota,
    compute_motp
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Working Directory set to: d:\tanne\Documents\cso_tomht


In [20]:
# --- PIPELINE CONFIGURATION ---
TARGET_TRAJECTORY_CSV = r"data\cso_data\Crossing.csv" # <-- UPDATE WITH TRAJECTORY CSV PATH
DETECTIONS_CSV_OUTPUT = "results/detections/master_detections_with_covariance.csv"
INPUT_FITS_DIR = "results/simulated_data/fits"
OUTPUT_CSV_DIR = "results/detections"
OUTPUT_PLOT_DIR = "results/detection_frames"

# Ensure the master results directory exists
os.makedirs("results", exist_ok=True)

In [21]:
def simulate_data(trajectory_csv: str):
    """Step 1: Generates synthetic FITS images from a trajectory CSV."""
    print("\n" + "="*40)
    print(" STEP 1: SIMULATING IMAGE DATA")
    print("="*40)
    
    # Call data simulation function
    simulate_fits_data(trajectory_csv, verbose=True)

# Execute Step 1
simulate_data(trajectory_csv=TARGET_TRAJECTORY_CSV)


 STEP 1: SIMULATING IMAGE DATA
[*] Loading trajectory: data\cso_data\Crossing.csv
[*] Running simulation logic...
[+] Simulation core complete. Generating artifacts...
[✓] Simulation successful. Outputs saved to: results/simulated_data


In [22]:
def detect_sources(input_dir: str, output_csv: str, output_plot: str):
    """Step 2: Runs background subtraction and matched filtering on the simulated FITS files."""
    print("\n" + "="*40)
    print(" STEP 2: SOURCE DETECTION (MATCHED FILTER)")
    print("="*40)
    
    process_fits_directory(
        input_dir=input_dir, 
        output_csv_dir=output_csv, 
        output_plot_dir=output_plot,
        sigma_psf=2.0, 
        threshold_factor=5.0,
        skip_bg_sub=False,
        verbose=True
    )

# Execute Step 2
detect_sources(
    input_dir=INPUT_FITS_DIR,
    output_csv=OUTPUT_CSV_DIR,
    output_plot=OUTPUT_PLOT_DIR
)


 STEP 2: SOURCE DETECTION (MATCHED FILTER)
[*] Starting Detection: 101 frames (USING BKG Subtraction)
[+] Processing Frame 101/101[✓] Saved 190 detections to CSV.
[*] Building animation...
[✓] GIF saved to results/detection_frames\detections_animation.gif


In [23]:
def run_tracker(csv_path: str, verbose: bool = True):
    """Step 3: Runs the TOMHT Tracker over the detections generated in Step 2."""
    print("\n" + "="*40)
    print(" STEP 3: TOMHT MULTI-TARGET TRACKING")
    print("="*40)
    
    output_dir = "results/tomht_eval/"
    os.makedirs(output_dir, exist_ok=True)
    
    raw_df = pd.read_csv(csv_path)
    raw_df = raw_df.rename(columns={'Frame_Idx': 'time', 'Centroid_X': 'x', 'Centroid_Y': 'y'})


    if verbose: print(f"[*] Running TOMHT over {len(raw_df)} pipeline detections...")
    
    config = TrackerConfig.from_jsonx("config/parameters.jsonx")
    tracker = TOMHTTracker(config)
    n_scan = config.n_scan_window 
    results = []
    
    # Tracking Loop
    unique_times = raw_df['time'].unique()
    for i, (t_raw, group) in enumerate(raw_df.groupby('time')):
        t = cast(int, t_raw) 
        meas_t = group[['x', 'y']].to_numpy()
        active_tracks = tracker.step(meas_t)

        for track in active_tracks:
            if len(track.best_hypothesis.history_states) >= n_scan: 
                delayed_state = track.best_hypothesis.history_states[-n_scan]
                results.append({
                    'time': t - n_scan,
                    'id': track.track_id,
                    'x': delayed_state[0], 'y': delayed_state[1],
                    'vx': delayed_state[2], 'vy': delayed_state[3]
                })
        
        if not verbose:
            print(f"\r[+] Tracking Progress: {i+1}/{len(unique_times)} frames", end='', flush=True)

    if not verbose: print()

    tracked_df = pd.DataFrame(results)
    
    if tracked_df.empty:
        print("[!] Warning: Tracker returned no tracks. Check gating or noise parameters.")
    else:
        if verbose:
            print_tomht_stats(raw_df, tracked_df)
            print("[*] Generating visualization artifacts...")
        
        plot_longest_tracks(raw_df, tracked_df, os.path.join(output_dir, "tomht_longest_tracks.png"))
        animate_tracks(raw_df, tracked_df, os.path.join(output_dir, "tomht_animation.gif"))
        
        if verbose: print(f"[✓] Tracker visualizations saved to: {output_dir}")

    return tracked_df

# Execute Step 3
tracked_df = run_tracker(csv_path=DETECTIONS_CSV_OUTPUT)

print("\n========================================")
print(" END-TO-END TOMHT TRACKING COMPLETE")
print("========================================")


 STEP 3: TOMHT MULTI-TARGET TRACKING
[*] Running TOMHT over 190 pipeline detections...
--------------------------------------------------
TOMHT PERFORMANCE SUMMARY
--------------------------------------------------
Frames Processed      : 101
Raw Detections        : 190
Total Tracked Points  : 236
Unique Tracks Formed  : 3
--------------------------------------------------
 TRACK LIFESPAN METRICS
--------------------------------------------------
Max Track Length      : 97 frames
Average Track Length  : 78.7 frames
Short Tracks (<=5)    : 0
Medium Tracks (6-20)  : 0
Long Tracks (>20)     : 3

[*] Generating visualization artifacts...
Saved track plot to results/tomht_eval/tomht_longest_tracks.png
Saved animation to results/tomht_eval/tomht_animation.gif
[✓] Tracker visualizations saved to: results/tomht_eval/

 END-TO-END TOMHT TRACKING COMPLETE


In [ ]:
# --- EVALUATION ---
GT_CSV_PATH = os.path.join(project_root, "results", "simulated_data", "ground_truth.csv")
gt_df = pd.read_csv(GT_CSV_PATH)

# Evaluate Tracking Performance
motp_score = compute_motp(gt_df, tracked_df, distance_threshold=5.0)
mota_score = compute_mota(gt_df, tracked_df, distance_threshold=5.0)

print(f"Tracking Performance:")
print(f" - MOTP (Precision): {motp_score:.3f} pixels average error")
print(f" - MOTA (Accuracy):  {mota_score:.3f}")

Tracking Performance:
 - MOTP (Precision): 0.401 pixels average error
 - MOTA (Accuracy):  0.624 (Max is 1.0)
